In [ ]:
replacement_dict = {
    '临夏': '临夏回族自治州',
    '伊犁': '伊犁哈萨克自治州',
    '克孜勒苏': '克孜勒苏柯尔克孜自治州',
    '临夏州': '临夏回族自治州',
    '临高': '临高县',
    '伊犁州': '伊犁哈萨克自治州',
    '克孜勒苏州': '克孜勒苏柯尔克孜自治州',
    '吉林市': '吉林',
    '大理': '大理白族自治州',
    '大理州': '大理白族自治州',
    '定安': '定安县',
    '屯昌': '屯昌县',
    '延边': '延边朝鲜族自治州',
    '德宏': '德宏傣族景颇族自治州',
    '怒江': '怒江傈僳族自治州',
    '果洛': '果洛藏族自治州',
    '楚雄': '楚雄彝族自治州',
    '海北': '海北藏族自治州',
    '海南': '海南藏族自治州',
    '海西': '海西蒙古族藏族自治州',
    '湘西': '湘西土家族苗族自治州',
    '德宏州': '德宏傣族景颇族自治州',
    '怒江州': '怒江傈僳族自治州',
    '果洛州': '果洛藏族自治州',
    '楚雄州': '楚雄彝族自治州',
    '海北州': '海北藏族自治州',
    '海南州': '海南藏族自治州',
    '海西州': '海西蒙古族藏族自治州',
    '湘西州': '湘西土家族苗族自治州',

    '澄迈': '澄迈县',
    '塔城':'塔城地区',
    '恩施':'恩施土家族苗族自治州',
    '凉山州':'凉山彝族自治州',
    
    '玉树': '玉树藏族自治州',
    '甘南': '甘南藏族自治州',
    '甘孜': '甘孜藏族自治州',
    '红河': '红河哈尼族彝族自治州',
    '迪庆': '迪庆藏族自治州',
    '阿坝': '阿坝藏族羌族自治州',
    '黄南': '黄南藏族自治州',
    '黔西南': '黔西南布依族苗族自治州',
    '昌吉':'昌吉回族自治州',
    '文山':'文山壮族苗族自治州',
    
    '玉树州': '玉树藏族自治州',
    '甘南州': '甘南藏族自治州',
    '甘孜州': '甘孜藏族自治州',
    '红河州': '红河哈尼族彝族自治州',
    '迪庆州': '迪庆藏族自治州',
    '阿坝州': '阿坝藏族羌族自治州',
    '黄南州': '黄南藏族自治州',
    '黔南州':'黔南布依族苗族自治州',
    '黔东南州': '黔东南苗族侗族自治州',
    '黔西南州': '黔西南布依族苗族自治州',
    '山南': '山南地区',
    '日喀则': '日喀则地区',
    '昌都': '昌都地区',
    '林芝': '林芝地区',
    '毕节': '毕节地区',
    '海东': '海东地区',
    '那曲': '那曲地区',
    '铜仁': '铜仁地区',
    '吐鲁番': '吐鲁番地区',
    '哈密':'哈密地区',
    '昌吉州':'昌吉回族自治州',
    '博尔塔拉州':'博尔塔拉蒙古自治州'
}

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv('./index2prop.csv', index_col=None)

In [ ]:
data

In [ ]:
import geopandas as gpd

# 指定 .shp 文件的完整路径
shp_path = r"E:\Data\heatwave\max_temp_national\2024.shp"

# 读取 shapefile
gdf = gpd.read_file(shp_path)

In [ ]:
gdf

In [ ]:
# 2. 只保留要素属性，去掉 geometry（如果之后还需要空间操作再把它合回去）
df = gdf.drop(columns='geometry')

# 3. 变形：把 20240101–20241231 这些列 melt 出来
df_long = df.melt(
    id_vars=['市代码', '市'],       # 保留这两列不动
    value_vars=[col for col in df.columns if col.isdigit()],  # 所有纯数字列
    var_name='date',
    value_name='temax'
)

# 4. 列重命名，并把 date 转成 datetime
df_long = (
    df_long
    .rename(columns={'市代码':'city_code', '市':'city_name'})
)
df_long['date'] = pd.to_datetime(df_long['date'], format='%Y%m%d')

# 5. 最终结果只保留四列
result = df_long[['city_code', 'city_name', 'date', 'temax']]

# 查看
print(result.head())

In [ ]:
result

In [ ]:
result['city_name'] = result['city_name'].str.replace('市$', '', regex=True)
result['city_name'] = result['city_name'].map(replacement_dict).fillna(result['city_name'])

In [ ]:
result.to_csv('2024_temax.csv', index=False, encoding='utf-8-sig')

In [ ]:
import pandas as pd

# 假设 merged 已经是你的 DataFrame，并且：
# merged.columns == ['city_code','city_name','date','temax']

# 1. 确保 date 列为 datetime，并按 city_name, date 排序
result['date'] = pd.to_datetime(result['date'])
result = result.sort_values(['city_name','date'])

In [ ]:
result

In [ ]:


# 2. 全年的完整日期索引
all_dates = pd.date_range('2024-01-01','2024-12-31', freq='D')

city_heatwave_encoding_2024 = {}

# 3. 按城市分组，逐个生成编码
for city, grp in result.groupby('city_name'):
    # 3.1 以全年日期重建序列，缺失的 temax 会变成 NaN
    s = grp.set_index('date')['temax'].reindex(all_dates)
    # 3.2 大于35度标记为 True，否则（包括 NaN）为 False
    mask = (s > 35).fillna(False)
    
    # 3.3 找出连续区间，run-length encoding
    encoding = [0] * len(mask)
    i = 0
    n = len(mask)
    while i < n:
        if mask.iloc[i]:
            # 如果当前位置为 True，就向后找整个连续 True 段
            j = i
            while j < n and mask.iloc[j]:
                j += 1
            run_len = j - i
            # 只有当长度>=3时，才算热浪，把整个区间都标 1
            if run_len >= 3:
                for k in range(i, j):
                    encoding[k] = 1
            # 跳到段尾
            i = j
        else:
            i += 1
    
    city_heatwave_encoding_2024[city] = encoding

# —— 检查几个城市的编码长度 —— #
for city, codes in list(city_heatwave_encoding_2024.items())[:5]:
    print(city, len(codes), codes[:10])  # 打印前 10 天的编码示例


In [ ]:
zhangzhou_encoding = city_heatwave_encoding_2024.get('漳州')

In [ ]:
zhangzhou_encoding

In [ ]:
heatwave_days = [i+1 for i, v in enumerate(beijing_encoding) if v == 1]

In [ ]:
heatwave_days

In [ ]:
import pandas as pd

# 假设你的原始表叫 result
# 1. 确保 date 列为 datetime，并排序
result['date'] = pd.to_datetime(result['date'])
result = result.sort_values(['city_name', 'date'])

# 2. 标记哪天 temax>35
result['above35'] = result['temax'] > 35

# 3. 对每个城市，给相同连续 True/False 段打上同一个组号
result['grp'] = result.groupby('city_name')['above35'] \
                      .transform(lambda x: x.ne(x.shift()).cumsum())

# 4. 只保留 temax>35 的那些段，聚合出每段的开始、结束和长度
runs = (
    result[result['above35']]
    .groupby(['city_name','grp'])
    .agg(start_date=('date','min'),
         end_date  =('date','max'),
         length    =('date','size'))
    .reset_index()
)

# 5. 只要长度>=3 的段，算作热浪
heatwave_runs = runs[runs['length'] >= 3]

# 6. 构造最终表，并重命名列
heatwave_periods_2024 = (
    heatwave_runs
    .loc[:, ['city_name','start_date','end_date']]
    .rename(columns={
        'city_name':'城市',
        'start_date':'开始日期',
        'end_date'  :'结束日期'
    })
    .sort_values(['城市','开始日期'])
    .reset_index(drop=True)
)

# 看一下结果
print(heatwave_periods_2024.head())

In [ ]:
heatwave_periods_2024[heatwave_periods_2024['城市'] == '漳州']

In [ ]:
result[result['city_name'] == '漳州'][180:190]

In [ ]:
heatwave_periods_2024.to_csv('heatwave_periods_2024.csv', index=False, encoding='utf-8-sig')

In [ ]:
import json

In [ ]:
out_path = r"C:\Users\Fe6ruar9\PhD\313.5-Heatwave\city_heatwave_encoding_2024.json"
with open(out_path, 'w', encoding='utf-8') as f:
    # ensure_ascii=False 保证中文键能正常写入
    json.dump(city_heatwave_encoding_2024, f, ensure_ascii=False, indent=2)

print(f"已保存到 {out_path}")

In [ ]:
result

In [ ]:
import pandas as pd
heatwave_periods_2024 = pd.read_csv(r'F:\PhD\313.5-Heatwave\2024\heatwave_periods_2024.csv', index_col=None, encoding='utf-8-sig')

In [ ]:
import pandas as pd

# 1. 模拟您的数据加载 (实际使用时请改为 pd.read_excel 或 pd.read_csv)

# 2. 确保日期列是 datetime 格式
heatwave_periods_2024['开始日期'] = pd.to_datetime(heatwave_periods_2024['开始日期'])
heatwave_periods_2024['结束日期'] = pd.to_datetime(heatwave_periods_2024['结束日期'])

# 3. 核心逻辑：将"时间段"炸裂(explode)成"每一天"
# 这一步是为了让跨月的长热浪（比如4月30日到5月2日）能正确地分别统计到4月和5月
all_days = []
for _, row in heatwave_periods_2024.iterrows():
    # 生成从开始到结束的所有日期
    days = pd.date_range(start=row['开始日期'], end=row['结束日期'])
    all_days.extend(days)

# 转换为 Series 方便统计
heatwave_series = pd.Series(all_days)

# 4. 统计分析
# 提取月份
monthly_counts = heatwave_series.dt.month.value_counts().sort_index()
total_days = len(heatwave_series)

# 创建结果表
result = pd.DataFrame({
    '热浪天数': monthly_counts,
    '占比(%)': (monthly_counts / total_days * 100).round(2)
})

# 计算累积占比（帮助判断覆盖度）
result['累积占比(%)'] = result['占比(%)'].cumsum()

print("------- 2024年热浪月份分布 -------")
print(result)

# 5. 自动判断集中区间 (例如覆盖95%天数的月份范围)
# 简单的逻辑：掐头去尾，或者找出占比最高的连续月份
threshold = 95.0
# 这里是一个简化的寻找逻辑：打印出占比较高的月份区间
high_impact_months = result[result['占比(%)'] > 1].index.tolist() # 忽略占比极小的月份
if high_impact_months:
    print(f"\n热浪主要集中在: {min(high_impact_months)}月 到 {max(high_impact_months)}月")

In [ ]:
heatwave_periods_2024

In [ ]:
# 确保开始和结束列是datetime格式
heatwave_periods_2024['开始日期'] = pd.to_datetime(heatwave_periods_2024['开始日期'])
heatwave_periods_2024['结束日期'] = pd.to_datetime(heatwave_periods_2024['结束日期'])

# 计算每个热浪事件的持续天数（包含首尾两天）
heatwave_periods_2024['持续时间'] = (heatwave_periods_2024['结束日期'] - heatwave_periods_2024['开始日期']).dt.days + 1

# 计算平均持续时间
average_duration = heatwave_periods_2024['持续时间'].mean()

print(f"所有城市热浪的平均持续时间为：{average_duration:.2f} 天")

In [ ]:
heatwave_periods_2024['持续时间'].sum()

In [ ]:
heatwave_periods_2024

In [ ]:
42525 * 6.5

In [ ]:
20500 *  6.5

In [ ]:
result_df.to_csv(r'.\SI_heatwave_periods_2024.csv', index=False, encoding='utf-8-sig')

In [ ]:
result_df = heatwave_periods_2024.groupby('城市')['持续时间'].sum().reset_index()

In [ ]:
result_df[result_df['持续时间'] == 30]

In [ ]:
result_df['城市'].unique()

In [ ]:
# 创建中文到英文的城市映射字典
city_mapping = {
    # 直辖市
    '北京': 'Beijing',
    '上海': 'Shanghai', 
    '天津': 'Tianjin',
    '重庆': 'Chongqing',
    
    # 省会城市
    '广州': 'Guangzhou',
    '南京': 'Nanjing',
    '杭州': 'Hangzhou',
    '成都': 'Chengdu',
    '西安': 'Xian',
    '武汉': 'Wuhan',
    '福州': 'Fuzhou',
    '南昌': 'Nanchang',
    '合肥': 'Hefei',
    '郑州': 'Zhengzhou',
    '济南': 'Jinan',
    '太原': 'Taiyuan',
    '石家庄': 'Shijiazhuang',
    '银川': 'Yinchuan',
    '乌鲁木齐': 'Urumqi',
    '长沙': 'Changsha',
    '南宁': 'Nanning',
    '海口': 'Haikou',
    
    # 其他主要城市
    '苏州': 'Suzhou',
    '无锡': 'Wuxi',
    '常州': 'Changzhou',
    '南通': 'Nantong',
    '扬州': 'Yangzhou',
    '镇江': 'Zhenjiang',
    '泰州': 'Taizhou',
    '宿迁': 'Suqian',
    '徐州': 'Xuzhou',
    '连云港': 'Lianyungang',
    '淮安': 'Huaian',
    '盐城': 'Yancheng',
    '平顶山': 'Pingdingshan',
    
    '宁波': 'Ningbo',
    '温州': 'Wenzhou',
    '嘉兴': 'Jiaxing',
    '湖州': 'Huzhou',
    '绍兴': 'Shaoxing',
    '金华': 'Jinhua',
    '衢州': 'Quzhou',
    '舟山': 'Zhoushan',
    '台州': 'Taizhou',
    '丽水': 'Lishui',
    
    '厦门': 'Xiamen',
    '莆田': 'Putian',
    '三明': 'Sanming',
    '泉州': 'Quanzhou',
    '漳州': 'Zhangzhou',
    '南平': 'Nanping',
    '龙岩': 'Longyan',
    '宁德': 'Ningde',
    
    '深圳': 'Shenzhen',
    '珠海': 'Zhuhai',
    '汕头': 'Shantou',
    '佛山': 'Foshan',
    '韶关': 'Shaoguan',
    '湛江': 'Zhanjiang',
    '肇庆': 'Zhaoqing',
    '江门': 'Jiangmen',
    '茂名': 'Maoming',
    '惠州': 'Huizhou',
    '梅州': 'Meizhou',
    '汕尾': 'Shanwei',
    '河源': 'Heyuan',
    '阳江': 'Yangjiang',
    '清远': 'Qingyuan',
    '东莞': 'Dongguan',
    '中山': 'Zhongshan',
    '潮州': 'Chaozhou',
    '揭阳': 'Jieyang',
    '云浮': 'Yunfu',
    
    # 继续其他城市...
    '大连': 'Dalian',
    '青岛': 'Qingdao',
    '烟台': 'Yantai',
    '威海': 'Weihai',
    '淄博': 'Zibo',
    '枣庄': 'Zaozhuang',
    '东营': 'Dongying',
    '潍坊': 'Weifang',
    '济宁': 'Jining',
    '泰安': 'Taian',
    '日照': 'Rizhao',
    '德州': 'Dezhou',
    '临沂': 'Linyi',
    '聊城': 'Liaocheng',
    '滨州': 'Binzhou',
    '菏泽': 'Heze',
    
    # 河南省
    '洛阳': 'Luoyang',
    '开封': 'Kaifeng',
    '新乡': 'Xinxiang',
    '焦作': 'Jiaozuo',
    '安阳': 'Anyang',
    '鹤壁': 'Hebi',
    '濮阳': 'Puyang',
    '许昌': 'Xuchang',
    '漯河': 'Luohe',
    '三门峡': 'Sanmenxia',
    '南阳': 'Nanyang',
    '商丘': 'Shangqiu',
    '信阳': 'Xinyang',
    '周口': 'Zhoukou',
    '驻马店': 'Zhumadian',
    '济源': 'Jiyuan',
    
    # 湖北省
    '宜昌': 'Yichang',
    '襄阳': 'Xiangyang',
    '荆州': 'Jingzhou',
    '荆门': 'Jingmen',
    '十堰': 'Shiyan',
    '孝感': 'Xiaogan',
    '黄冈': 'Huanggang',
    '鄂州': 'Ezhou',
    '黄石': 'Huangshi',
    '咸宁': 'Xianning',
    '随州': 'Suizhou',
    '恩施土家族苗族自治州': 'Enshi',
    '仙桃': 'Xiantao',
    '潜江': 'Qianjiang',
    '天门': 'Tianmen',
    '神农架林区': 'Shennongjia',
    
    # 湖南省
    '株洲': 'Zhuzhou',
    '湘潭': 'Xiangtan',
    '衡阳': 'Hengyang',
    '邵阳': 'Shaoyang',
    '岳阳': 'Yueyang',
    '常德': 'Changde',
    '张家界': 'Zhangjiajie',
    '益阳': 'Yiyang',
    '郴州': 'Chenzhou',
    '永州': 'Yongzhou',
    '怀化': 'Huaihua',
    '娄底': 'Loudi',
    '湘西土家族苗族自治州': 'Xiangxi',
    
    # 四川省
    '绵阳': 'Mianyang',
    '德阳': 'Deyang',
    '南充': 'Nanchong',
    '广安': 'Guangan',
    '遂宁': 'Suining',
    '内江': 'Neijiang',
    '乐山': 'Leshan',
    '宜宾': 'Yibin',
    '广元': 'Guangyuan',
    '泸州': 'Luzhou',
    '达州': 'Dazhou',
    '巴中': 'Bazhong',
    '雅安': 'Yaan',
    '眉山': 'Meishan',
    '资阳': 'Ziyang',
    '自贡': 'Zigong',
    '攀枝花': 'Panzhihua',
    
    # 陕西省
    '宝鸡': 'Baoji',
    '咸阳': 'Xianyang',
    '铜川': 'Tongchuan',
    '渭南': 'Weinan',
    '延安': 'Yanan',
    '榆林': 'Yulin',
    '汉中': 'Hanzhong',
    '安康': 'Ankang',
    '商洛': 'Shangluo',
    
    # 安徽省
    '芜湖': 'Wuhu',
    '蚌埠': 'Bengbu',
    '淮南': 'Huainan',
    '马鞍山': 'Maanshan',
    '淮北': 'Huaibei',
    '铜陵': 'Tongling',
    '安庆': 'Anqing',
    '黄山': 'Huangshan',
    '滁州': 'Chuzhou',
    '阜阳': 'Fuyang',
    '宿州': 'Suzhou',
    '巢湖': 'Chaohu',
    '六安': 'Luan',
    '亳州': 'Bozhou',
    '池州': 'Chizhou',
    '宣城': 'Xuancheng',
    
    # 江西省
    '九江': 'Jiujiang',
    '上饶': 'Shangrao',
    '抚州': 'Fuzhou',
    '宜春': 'Yichun',
    '吉安': 'Jian',
    '赣州': 'Ganzhou',
    '景德镇': 'Jingdezhen',
    '萍乡': 'Pingxiang',
    '新余': 'Xinyu',
    '鹰潭': 'Yingtan',
    
    # 广西
    '柳州': 'Liuzhou',
    '桂林': 'Guilin',
    '梧州': 'Wuzhou',
    '北海': 'Beihai',
    '防城港': 'Fangchenggang',
    '钦州': 'Qinzhou',
    '贵港': 'Guigang',
    '玉林': 'Yulin',
    '百色': 'Baise',
    '贺州': 'Hezhou',
    '河池': 'Hechi',
    '来宾': 'Laibin',
    '崇左': 'Chongzuo',
    
    # 海南省
    '三亚': 'Sanya',
    '儋州': 'Danzhou',
    '文昌': 'Wenchang',
    '琼海': 'Qionghai',
    '万宁': 'Wanning',
    '东方': 'Dongfang',
    '定安县': 'Dingan',
    '屯昌县': 'Tunchang',
    '澄迈县': 'Chengmai',
    '临高县': 'Lingao',
    '白沙黎族自治县': 'Baisha',
    '昌江黎族自治县': 'Changjiang',
    '琼中黎族苗族自治县': 'Qiongzhong',
    
    # 山西省
    '大同': 'Datong',
    '阳泉': 'Yangquan',
    '长治': 'Changzhi',
    '晋城': 'Jincheng',
    '朔州': 'Shuozhou',
    '晋中': 'Jinzhong',
    '运城': 'Yuncheng',
    '忻州': 'Xinzhou',
    '临汾': 'Linfen',
    '吕梁': 'Lvliang',
    
    # 河北省
    '唐山': 'Tangshan',
    '秦皇岛': 'Qinhuangdao',
    '邯郸': 'Handan',
    '邢台': 'Xingtai',
    '保定': 'Baoding',
    '张家口': 'Zhangjiakou',
    '承德': 'Chengde',
    '沧州': 'Cangzhou',
    '廊坊': 'Langfang',
    '衡水': 'Hengshui',
    
    # 内蒙古
    '包头': 'Baotou',
    '乌海': 'Wuhai',
    '赤峰': 'Chifeng',
    '通辽': 'Tongliao',
    '鄂尔多斯': 'Ordos',
    '巴彦淖尔': 'Bayannur',
    '乌兰察布': 'Ulanqab',
    '兴安盟': 'Xingan',
    '锡林郭勒盟': 'Xilingol',
    '阿拉善盟': 'Alasha',
    
    # 新疆
    '克拉玛依': 'Karamay',
    '吐鲁番地区': 'Turpan',
    '哈密地区': 'Hami',
    '昌吉回族自治州': 'Changji',
    '博尔塔拉蒙古自治州': 'Bortala',
    '巴音郭楞蒙古自治州': 'Bayingolin',
    '阿克苏地区': 'Aksu',
    '喀什地区': 'Kashgar',
    '和田地区': 'Hotan',
    '伊犁哈萨克自治州': 'Ili',
    '塔城地区': 'Tacheng',
    '阿勒泰地区': 'Altay',
    '石河子': 'Shihezi',
    '阿拉尔': 'Aral',
    '图木舒克': 'Tumxuk',
    '五家渠': 'Wujiaqu',
    '铁门关': 'Tiemenguan',
    '双河': 'Shuanghe',
    '可克达拉': 'Kokdala',
    '昆玉': 'Kunyu',
    '胡杨河': 'Huyanghe',
    '新星': 'Xinxing',
    '白杨': 'Baiyang',
    
    # 宁夏
    '石嘴山': 'Shizuishan',
    '吴忠': 'Wuzhong',
    '固原': 'Guyuan',
    '中卫': 'Zhongwei',
    
    # 贵州
    '六盘水': 'Liupanshui',
    '遵义': 'Zunyi',
    '安顺': 'Anshun',
    '铜仁地区': 'Tongren',
    '黔西南': 'Qianxinan',
    '毕节': 'Bijie',
    '黔东南苗族侗族自治州': 'Qiandongnan',
    '黔南布依族苗族自治州': 'Qiannan',
    
    # 云南
    '曲靖': 'Qujing',
    '玉溪': 'Yuxi',
    '保山': 'Baoshan',
    '昭通': 'Zhaotong',
    '丽江': 'Lijiang',
    '思茅': 'Simao',
    '临沧': 'Lincang',
    '楚雄': 'Chuxiong',
    '红河': 'Honghe',
    '文山': 'Wenshan',
    '西双版纳傣族自治州': 'Xishuangbanna',
    '大理': 'Dali',
    '德宏': 'Dehong',
    '怒江': 'Nujiang',
    '迪庆': 'Diqing',
    
    # 辽宁
    '沈阳': 'Shenyang',
    '大连': 'Dalian',
    '鞍山': 'Anshan',
    '抚顺': 'Fushun',
    '本溪': 'Benxi',
    '丹东': 'Dandong',
    '锦州': 'Jinzhou',
    '营口': 'Yingkou',
    '阜新': 'Fuxin',
    '辽阳': 'Liaoyang',
    '盘锦': 'Panjin',
    '铁岭': 'Tieling',
    '朝阳': 'Chaoyang',
    '葫芦岛': 'Huludao'
}

# 应用映射
result_df['城市'] = result_df['城市'].map(city_mapping)

In [ ]:
result_df

In [ ]:
unmapped_cities = result_df[result_df['城市'].isna()]
if not unmapped_cities.empty:
    print("未映射的城市：")
    print(unmapped_cities)